# 02a – Cleaning the Bicycle Traffic Data

## Objective
The bicycle traffic data from the City of Zurich for the years
2023 to 2025 will be loaded, examined, cleaned,
and aggregated on an hourly basis.

## Procedure
1. Combine the annual datasets
2. Examine data quality
3. Select bicycle traffic measurements
4. Handle invalid entries
5. Calculate hourly values
6. Save the cleaned dataset

In [ ]:
import pandas as pd
from pathlib import Path

# find Proect root
cwd = Path.cwd()

project_root = next(
    p for p in [cwd, *cwd.parents]
    if (p / "data").is_dir()
    and (p / "notebooks").is_dir()
)

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print("Rohdaten:", raw_dir)
print("Bereinigte Daten:", processed_dir)

Rohdaten: c:\Projects\velo-wetter-zuerich-HS26\data\raw
Bereinigte Daten: c:\Projects\velo-wetter-zuerich-HS26\data\processed


In [2]:
years = [2023, 2024, 2025]

dataframes = []

for year in years:

    file_path = raw_dir / f"velo_{year}.csv"

    df_year = pd.read_csv(file_path)

    df_year["QUELLJAHR"] = year

    dataframes.append(df_year)

    print(
        f"{year}: {len(df_year):,} Zeilen geladen"
    )

# Alle Jahresdaten zusammenführen
df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nGesamtanzahl Zeilen:", len(df))

display(df.head())

2023: 949,660 Zeilen geladen
2024: 918,210 Zeilen geladen
2025: 828,507 Zeilen geladen

Gesamtanzahl Zeilen: 2696377


,FK_STANDORT,DATUM,VELO_IN,VELO_OUT,FUSS_IN,FUSS_OUT,OST,NORD,QUELLJAHR
0,5003,2023-01-01T00:00,NaN,NaN,1.0,0.0,2682978,1248744,2023
1,4257,2023-01-01T00:00,0.0,0.0,NaN,NaN,2681857,1251991,2023
2,394,2023-01-01T00:00,NaN,NaN,1.0,1.0,2683573,1251687,2023
3,2986,2023-01-01T00:00,0.0,0.0,NaN,NaN,2684578,1251966,2023
4,3598,2023-01-01T00:00,0.0,0.0,NaN,NaN,2684006,1246566,2023


In [3]:
# Überblick über die Datentypen
df.info()

# Fehlende Werte
display(df.isna().sum())

# Doppelte Standort-Zeit-Kombinationen
print(
    "Duplikate:",
    df.duplicated(
        subset=["FK_STANDORT", "DATUM"]
    ).sum()
)

# Anzahl Zählstellen
print(
    "Zählstellen:",
    df["FK_STANDORT"].nunique()
)

<class 'pandas.DataFrame'>
RangeIndex: 2696377 entries, 0 to 2696376
Data columns (total 9 columns):
 #   Column       Dtype  
---  ------       -----  
 0   FK_STANDORT  int64  
 1   DATUM        str    
 2   VELO_IN      float64
 3   VELO_OUT     float64
 4   FUSS_IN      float64
 5   FUSS_OUT     float64
 6   OST          int64  
 7   NORD         int64  
 8   QUELLJAHR    int64  
dtypes: float64(4), int64(4), str(1)
memory usage: 185.1 MB


FK_STANDORT          0
DATUM                0
VELO_IN         344382
VELO_OUT        619494
FUSS_IN        2351995
FUSS_OUT       2351995
OST                  0
NORD                 0
QUELLJAHR            0
dtype: int64

Duplikate: 296
Zählstellen: 48


In [4]:
# Zeilen mit mindestens einem Velomesswert
velo_mask = df[
    ["VELO_IN", "VELO_OUT"]
].notna().any(axis=1)

velo = df.loc[velo_mask].copy()

print("Ursprüngliche Zeilen:", len(df))
print("Zeilen mit Velomessung:", len(velo))
print("Ausgeschlossene Zeilen:", len(df) - len(velo))

Ursprüngliche Zeilen: 2696377
Zeilen mit Velomessung: 2351995
Ausgeschlossene Zeilen: 344382


In [5]:
# Datumswerte konvertieren
velo["DATUM"] = pd.to_datetime(
    velo["DATUM"],
    errors="coerce"
)

# Velowerte numerisch konvertieren
for col in ["VELO_IN", "VELO_OUT"]:
    velo[col] = pd.to_numeric(
        velo[col],
        errors="coerce"
    )

# Ungültige Daten zählen
invalid_dates = velo["DATUM"].isna().sum()

print("Ungültige Datumswerte:", invalid_dates)

# Ungültige Einträge ausschliessen
velo = velo.dropna(
    subset=["DATUM", "FK_STANDORT"]
)

# Nur Werte im Untersuchungszeitraum
velo = velo[
    velo["DATUM"].dt.year.isin(years)
].copy()

print("Verbleibende Zeilen:", len(velo))

Ungültige Datumswerte: 0
Verbleibende Zeilen: 2351995


In [6]:
# Uhrzeit auf volle Stunde abrunden
velo["stunde"] = velo["DATUM"].dt.floor("h")

# Pro Standort und Stunde zusammenfassen
velo_stunden = (
    velo.groupby(
        ["FK_STANDORT", "stunde"],
        as_index=False
    )
    .agg(
        velo_in=(
            "VELO_IN",
            lambda x: x.sum(min_count=1)
        ),
        velo_out=(
            "VELO_OUT",
            lambda x: x.sum(min_count=1)
        ),
        anzahl_intervalle=("DATUM", "size"),
        messungen_in=("VELO_IN", "count"),
        messungen_out=("VELO_OUT", "count")
    )
)

# Verfügbare Richtungswerte zusammenzählen
velo_stunden["velo_total"] = velo_stunden[
    ["velo_in", "velo_out"]
].sum(axis=1, min_count=1)

display(velo_stunden.head())

,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total
0,60,2023-01-01 00:00:00,0.0,1.0,4,4,4,1.0
1,60,2023-01-01 01:00:00,0.0,2.0,4,4,4,2.0
2,60,2023-01-01 02:00:00,0.0,3.0,4,4,4,3.0
3,60,2023-01-01 03:00:00,2.0,2.0,4,4,4,4.0
4,60,2023-01-01 04:00:00,0.0,0.0,4,4,4,0.0


In [7]:
output_path = (
    processed_dir / "velo_hourly_2023_2025.csv"
)

velo_stunden.to_csv(
    output_path,
    index=False
)

print("Datei gespeichert:", output_path)

print("Anzahl Stunden-Zählstellen:", len(velo_stunden))
print("Erste Stunde:", velo_stunden["stunde"].min())
print("Letzte Stunde:", velo_stunden["stunde"].max())

Datei gespeichert: c:\Projects\velo-wetter-zuerich-HS26\data\processed\velo_hourly_2023_2025.csv
Anzahl Stunden-Zählstellen: 587954
Erste Stunde: 2023-01-01 00:00:00
Letzte Stunde: 2025-12-31 23:00:00
